In [6]:
import pandas as pd
from google.colab import drive
import os

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 修改讀取路徑：指向雲端硬碟裡的 Colab Notebooks
# 請確認檔名 task1.xlsx 是否正確
file_path = "/content/drive/MyDrive/Colab Notebooks/hw1.xlsx"

if os.path.exists(file_path):
    df = pd.read_excel(file_path)

    # 3. 命名邏輯：補零、底線、加副檔名
    # 使用 pd.to_datetime 自動處理日期格式
    df['File'] = 'OptionsDaily_' + pd.to_datetime(df['年月日']).dt.strftime('%Y_%m_%d') + '.csv'

    # 4. 修改儲存路徑：存回雲端硬碟同一個資料夾
    output_path = "/content/drive/MyDrive/Colab Notebooks/hw1_updated.csv"

    # 5. 儲存檔案
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("✅ 處理完成！")
    print("產生的 File 範例：", df['File'].iloc[0])
    print(f"檔案已儲存至：{output_path}")
else:
    print(f"❌ 找不到檔案！請檢查 Google Drive 的 'Colab Notebooks' 資料夾裡是否有 'hw1.xlsx'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 處理完成！
產生的 File 範例： OptionsDaily_2021_01_04.csv
檔案已儲存至：/content/drive/MyDrive/Colab Notebooks/hw1_updated.csv


In [7]:
import pandas as pd
from google.colab import drive
import datetime
import os

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 設定路徑 (從雲端硬碟讀取 task1.xlsx)
file_path = "/content/drive/MyDrive/Colab Notebooks/hw1.xlsx"
output_path = "/content/drive/MyDrive/Colab Notebooks/hw1_final.csv"

# 定義計算「每月第三個星期三」的函數
def get_third_wednesday(year, month):
    first_day = datetime.date(year, month, 1)
    # weekday: 0=Mon, 2=Wed
    first_wednesday = (2 - first_day.weekday() + 7) % 7
    return first_day + datetime.timedelta(days=first_wednesday + 14)

if os.path.exists(file_path):
    # 3. 讀取並更名基礎欄位
    df = pd.read_excel(file_path)

    # 清理欄位名稱並對應英文
    df.columns = df.columns.astype(str).str.strip()
    rename_dict = {
        '年月日': 'Date',
        '收盤價(元)': 'S0',
        'date': 'Date',   # 相容不同可能的原始名稱
        '日期': 'Date'
    }
    df.rename(columns=rename_dict, inplace=True)

    # 確保 Date 是 datetime 格式
    df['Date'] = pd.to_datetime(df['Date'])

    # 4. 生成 File 欄位 (格式: OptionsDaily_YYYY_MM_DD.csv)
    df['File'] = 'OptionsDaily_' + df['Date'].dt.strftime('%Y_%m_%d') + '.csv'

    # 5. 計算 Maturity, contract, ContractExpiryDate
    contracts, expiry_dates, maturities = [], [], []

    for idx, row in df.iterrows():
        current_date = row['Date'].date()

        # 找出當月結算日
        this_month_expiry = get_third_wednesday(current_date.year, current_date.month)

        # 條件：距離結算日需大於 1 天，否則換下個月
        if (this_month_expiry - current_date).days > 1:
            target_expiry = this_month_expiry
        else:
            nm = current_date.month + 1 if current_date.month < 12 else 1
            ny = current_date.year if current_date.month < 12 else current_date.year + 1
            target_expiry = get_third_wednesday(ny, nm)

        contracts.append(target_expiry.strftime('%Y%m'))
        expiry_dates.append(target_expiry)
        maturities.append((target_expiry - current_date).days)

    # 新增計算出的欄位
    df['contract'] = contracts
    df['ContractExpiryDate'] = expiry_dates
    df['Maturity'] = maturities

    # 6. 最後整理：將日期轉回字串，並依照要求的順序排列欄位
    df['Date'] = df['Date'].dt.strftime('%Y/%m/%d')

    # --- 調整後的欄位順序：Maturity 放在 S0 與 contract 中間 ---
    final_cols = ['Date', 'File', 'S0', 'Maturity', 'contract', 'ContractExpiryDate']
    df = df[final_cols]

    # 7. 儲存至雲端硬碟
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("✅ 處理完成！欄位順序已調整。")
    print(df.head())
    print(f"\n新檔案已儲存：{output_path}")

else:
    print(f"❌ 找不到檔案：{file_path}，請確認檔案已上傳至雲端硬碟。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 處理完成！欄位順序已調整。
         Date                         File        S0  Maturity contract  \
0  2021/01/04  OptionsDaily_2021_01_04.csv  14902.03        16   202101   
1  2021/01/05  OptionsDaily_2021_01_05.csv  15000.03        15   202101   
2  2021/01/06  OptionsDaily_2021_01_06.csv  14983.13        14   202101   
3  2021/01/07  OptionsDaily_2021_01_07.csv  15214.00        13   202101   
4  2021/01/08  OptionsDaily_2021_01_08.csv  15463.95        12   202101   

  ContractExpiryDate  
0         2021-01-20  
1         2021-01-20  
2         2021-01-20  
3         2021-01-20  
4         2021-01-20  

新檔案已儲存：/content/drive/MyDrive/Colab Notebooks/hw1_final.csv


In [3]:
#2020
import pandas as pd
from google.colab import drive
import os

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 設定路徑 (讀取你之前的 final 檔案)
file_path = "/content/drive/MyDrive/Colab Notebooks/hw1_final.csv"
output_path = "/content/drive/MyDrive/Colab Notebooks/hw1_final_2020_RF.csv"

if os.path.exists(file_path):
    # 3. 讀取資料
    df = pd.read_csv(file_path)

    # 確保 Date 格式正確
    df['Date'] = pd.to_datetime(df['Date'])

    # 4. 建立 2020 年精準利率變動表
    # 資料來源：台灣銀行歷史牌告利率 (一年期定儲、一般、機動)
    rf_history = [
        {'start_date': '1900-01-01', 'rate': 1.090}, # 預設起始值 (2020年初適用)
        {'start_date': '2020-03-23', 'rate': 0.840}, # 2020/3/23 降息一碼
    ]

    rf_df = pd.DataFrame(rf_history)
    rf_df['start_date'] = pd.to_datetime(rf_df['start_date'])

    # 5. 定義比對函數
    def get_2020_rf(trade_date):
        # 找出在交易日之前最後一次的利率調整紀錄
        applicable_rates = rf_df[rf_df['start_date'] <= trade_date]
        # 轉換為小數 (例如 0.84% -> 0.0084)
        return applicable_rates.iloc[-1]['rate'] / 100

    # 6. 執行計算並新增 RF 欄位
    df['RF'] = df['Date'].apply(get_2020_rf)

    # 7. 格式整理：保持 Date 格式為 YYYY/MM/DD
    df['Date'] = df['Date'].dt.strftime('%Y/%m/%d')

    # 8. 儲存檔案
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("✅ 2020 年 RF 利率已成功對應！")
    print("-" * 30)
    print("【數據驗證】")
    print(f"3/23 降息前範例：\n{df[df['Date'] < '2020/03/23'][['Date', 'RF']].head(2)}")
    print(f"\n3/23 降息後範例：\n{df[df['Date'] >= '2020/03/23'][['Date', 'RF']].head(2)}")
    print("-" * 30)
    print(f"檔案已儲存：{output_path}")

else:
    print(f"❌ 找不到檔案：{file_path}，請確認檔名是否正確。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 2020 年 RF 利率已成功對應！
------------------------------
【數據驗證】
3/23 降息前範例：
         Date      RF
0  2020/01/02  0.0109
1  2020/01/03  0.0109

3/23 降息後範例：
          Date      RF
49  2020/03/23  0.0084
50  2020/03/24  0.0084
------------------------------
檔案已儲存：/content/drive/MyDrive/Colab Notebooks/hw1_final_2020_RF.csv


In [8]:
import pandas as pd
from google.colab import drive
import os

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 設定路徑
file_path = "/content/drive/MyDrive/Colab Notebooks/hw1_final.csv"
output_path = "/content/drive/MyDrive/Colab Notebooks/hw1_final_2021_RF.csv"

if os.path.exists(file_path):
    # 3. 讀取資料
    df = pd.read_csv(file_path)

    # 確保 Date 格式正確
    df['Date'] = pd.to_datetime(df['Date'])

    # 4. 2021 年利率邏輯：
    # 根據台銀歷史資料，2021 全年度利率皆為 0.840%
    # 我們直接設定為 0.0084 (小數格式)
    df['RF'] = 0.0084

    # 5. 格式整理：保持 Date 格式為 YYYY/MM/DD
    df['Date'] = df['Date'].dt.strftime('%Y/%m/%d')

    # 6. 欄位排序 (保持你之前的要求：Maturity 在 S0 與 contract 中間)
    # 確保 RF 放在最後或指定位置，這裡建議放在最後作為參數
    cols = ['Date', 'File', 'S0', 'Maturity', 'contract', 'ContractExpiryDate', 'RF']
    df = df[cols]

    # 7. 儲存檔案
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("✅ 2021 年 RF 利率已成功填入！")
    print("-" * 30)
    print(f"【數據檢查】")
    print(f"利率設定值：0.84% (程式記錄為 0.0084)")
    print(df[['Date', 'S0', 'Maturity', 'RF']].head())
    print("-" * 30)
    print(f"檔案已儲存：{output_path}")

else:
    print(f"❌ 找不到檔案：{file_path}，請確認雲端硬碟路徑。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 2021 年 RF 利率已成功填入！
------------------------------
【數據檢查】
利率設定值：0.84% (程式記錄為 0.0084)
         Date        S0  Maturity      RF
0  2021/01/04  14902.03        16  0.0084
1  2021/01/05  15000.03        15  0.0084
2  2021/01/06  14983.13        14  0.0084
3  2021/01/07  15214.00        13  0.0084
4  2021/01/08  15463.95        12  0.0084
------------------------------
檔案已儲存：/content/drive/MyDrive/Colab Notebooks/hw1_final_2021_RF.csv
